In [1]:


import os

# Specify the root directory where PDF files exist
root_directory = "/home/abusufyan/development/data"

# Set the batch size (number of files to process in each batch)
batch_size = 100

# Initialize an empty list to store loaded documents
docs = []

# Function to process a batch of PDF files
def process_pdf_batch(pdf_files):
    batch_docs = []
    for pdf_file_path in pdf_files:
        pdf_loader = PyPDFLoader(pdf_file_path)
        batch_docs.extend(pdf_loader.load())
    return batch_docs

# Get the list of PDF files to process
pdf_files_to_process = []
for root, dirs, files in os.walk(root_directory):
    pdf_files_to_process.extend([os.path.join(root, file) for file in files if file.lower().endswith(".pdf")])

# Create a ThreadPoolExecutor for parallel processing
with ThreadPoolExecutor() as executor:
    total_files = len(pdf_files_to_process)
    processed_files = 0

    # Iterate through the PDF files in batches
    for i in range(0, total_files, batch_size):
        batch = pdf_files_to_process[i:i+batch_size]
        batch_docs = list(executor.map(process_pdf_batch, [batch]))
        for batch_result in batch_docs:
            docs.extend(batch_result)
            processed_files += len(batch)
            print(f"Processed {processed_files} / {total_files} files")

# Now 'docs' contains the loaded documents from all the PDF files found in the 'data' directory and its subdirectories.

Processed 100 / 48375 files
Processed 200 / 48375 files
Processed 300 / 48375 files
Processed 400 / 48375 files
Processed 500 / 48375 files
Processed 600 / 48375 files
Processed 700 / 48375 files
Processed 800 / 48375 files
Processed 900 / 48375 files
Processed 1000 / 48375 files
Processed 1100 / 48375 files
Processed 1200 / 48375 files
Processed 1300 / 48375 files
Processed 1400 / 48375 files
Processed 1500 / 48375 files
Processed 1600 / 48375 files
Processed 1700 / 48375 files
Processed 1800 / 48375 files
Processed 1900 / 48375 files
Processed 2000 / 48375 files
Processed 2100 / 48375 files
Processed 2200 / 48375 files
Processed 2300 / 48375 files
Processed 2400 / 48375 files
Processed 2500 / 48375 files
Processed 2600 / 48375 files
Processed 2700 / 48375 files
Processed 2800 / 48375 files
Processed 2900 / 48375 files
Processed 3000 / 48375 files
Processed 3100 / 48375 files
Processed 3200 / 48375 files
Processed 3300 / 48375 files
Processed 3400 / 48375 files
Processed 3500 / 48375 

In [2]:
#import necessary libraries
import os
import openai
from langchain.prompts import PromptTemplate
from langchain.vectorstores import Chroma
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings.openai import OpenAIEmbeddings
from dotenv import load_dotenv, find_dotenv
from langchain.document_loaders import PyPDFLoader
from concurrent.futures import ThreadPoolExecutor
from langchain.chat_models import ChatOpenAI
from langchain.chains import RetrievalQA

In [9]:


# Load environment variables from the .env file
load_dotenv(find_dotenv())

# Access the environment variable
openai.api_key = os.environ['OPENAI_API_KEY']


import datetime
current_date = datetime.datetime.now().date()
if current_date < datetime.date(2023, 9, 2):
    llm_name = "gpt-3.5-turbo-0301"
else:
    llm_name = "gpt-3.5-turbo"
print(llm_name)

llm = ChatOpenAI(model_name = llm_name, temperature=0)

gpt-3.5-turbo


# length of 'docs'

In [ ]:
print("Total pages are :",len(docs))

# Load docs object

In [141]:
# To load the variable back later:
import pickle
with open('docs_data.pkl', 'rb') as file:
    docs = pickle.load(file)

print("Total pages are :",len(docs))

Total pages are : 121612


In [ ]:
#import libraries
from langchain.vectorstores import Chroma
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings.openai import OpenAIEmbeddings


# Initialize the text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=300
)

# Split the list of documents into smaller batches
batch_size = 200  # Choose an appropriate batch size
document_batches = [docs[i:i + batch_size] for i in range(0, len(docs), batch_size)]

# Initialize embedding using OpenAIEmbeddings class
embedding = OpenAIEmbeddings(show_progress_bar=True)

persist_directory = 'updated_docs/chroma'  # Specify the common root directory

# Iterate through document batches and process them
for i, document_batch in enumerate(document_batches):
    splits = text_splitter.split_documents(document_batch)
    
    # Creating Embeddings & Store into VectorDB for each batch
    vectordb = Chroma.from_documents(
        documents=splits,
        embedding=embedding,
        persist_directory=persist_directory
    )


# Load VectorDB from local disk

In [104]:
# Initialize embedding using OpenAIEmbeddings class
embedding = OpenAIEmbeddings(show_progress_bar=True)
vectordb = Chroma(persist_directory="docs/chroma", embedding_function=embedding)

In [105]:
vectordb._collection.count()

476532

In [115]:
question = "what is guinea pig"

# Similarity Search

In [ ]:
docs = vectordb.similarity_search(question,k=5)
docs

# Build Prompt Template

In [111]:
from langchain.prompts import PromptTemplate

# Build prompt
template = """Use the following pieces of context to answer the question at the end. Keep the answer as concise as possible. Always say "thanks for asking!" at the end of the answer. 
{context}
Question: {question}
Helpful Answer:"""
QA_CHAIN_PROMPT = PromptTemplate.from_template(template)

# RetrievalQA Chain

In [112]:
# Run chain
qa_chain = RetrievalQA.from_chain_type(
    llm,
    retriever=vectordb.as_retriever(),
    return_source_documents=True,
    chain_type_kwargs={"prompt": QA_CHAIN_PROMPT}
)

In [113]:
qa_chain({"query": question})

  0%|          | 0/1 [00:00<?, ?it/s]

{'query': 'What is aggression in horses, and how is it primarily used as a form of communication?',
 'result': 'Aggression in horses is a form of communication used to establish precedence and consists of threats or harmful actions directed towards an individual. Thanks for asking!',
 'source_documents': [Document(page_content='to tell the di\x00erence\n©Vetstream Ltd\nWhat is aggression?\nAggression is primarily a form of communication used to establish precedence and consists of\nthreats or harmful actions directed towards an individual. Horses in the wild show very little overt\nphysical aggression as they normally live in stable social groups. When they do occur, aggressive\nencounters are normally short-lived and end with one individual retreating away from the\nsituation. Broadly speaking aggression occurs when a horse perceives some form of threat to', metadata={'page': 0, 'source': '/home/abusufyan/development/data/EQUIS/Media/Behavior/Factsheet/Aggression.pdf'}),
  Document(pa

# Custom Prompt (ConversationalQAChain)

In [52]:
from langchain.prompts.prompt import PromptTemplate
custom_template = """Given the following conversation and a follow up question, rephrase the follow up question to be a standalone question. And then give answer according to it.
Chat History:
{chat_history}
Follow Up Input: {question}
Standalone question:"""

CUSTOM_QUESTION_PROMPT = PromptTemplate.from_template(custom_template)

# Memory

In [53]:
from langchain.memory import ConversationBufferMemory
memory = ConversationBufferMemory(
    memory_key= "chat_history",
    return_messages= True
)

# ConversationalRetrievalChain

In [54]:
from langchain.chains import ConversationalRetrievalChain
Retriever  = vectordb.as_retriever()

crc_qa = ConversationalRetrievalChain.from_llm(
    llm,
    retriever=Retriever,
    memory=memory,
    condense_question_prompt=CUSTOM_QUESTION_PROMPT
    
)

In [55]:
crc_qa({"question": question})

  0%|          | 0/1 [00:00<?, ?it/s]

{'question': 'What are the key clinical effects of rotavirus infection in cattle, and how does the virus lead to these effects on the digestive system and overall health of the animals?',
 'chat_history': [HumanMessage(content='What are the key clinical effects of rotavirus infection in cattle, and how does the virus lead to these effects on the digestive system and overall health of the animals?'),
  AIMessage(content='The key clinical effects of rotavirus infection in cattle include diarrhea, dehydration, and decreased appetite. The virus primarily affects the small intestine, specifically the enterocytes on the villi. Upon infection, the virus destroys these enterocytes, leading to a reduction in disaccharide levels and defective glucose coupled sodium transport. This impairs the absorption of nutrients and electrolytes from the intestine, resulting in diarrhea and fluid retention in the lumen. Additionally, the non-structural protein NSP4 acts as an enterotoxin, further inhibiting 

In [56]:
crc_qa({"question": question})

  0%|          | 0/1 [00:00<?, ?it/s]

{'question': 'What are the key clinical effects of rotavirus infection in cattle, and how does the virus lead to these effects on the digestive system and overall health of the animals?',
 'chat_history': [HumanMessage(content='What are the key clinical effects of rotavirus infection in cattle, and how does the virus lead to these effects on the digestive system and overall health of the animals?'),
  AIMessage(content='The key clinical effects of rotavirus infection in cattle include diarrhea, dehydration, and decreased appetite. The virus primarily affects the small intestine, specifically the enterocytes on the villi. Upon infection, the virus destroys these enterocytes, leading to a reduction in disaccharide levels and defective glucose coupled sodium transport. This impairs the absorption of nutrients and electrolytes from the intestine, resulting in diarrhea and fluid retention in the lumen. Additionally, the non-structural protein NSP4 acts as an enterotoxin, further inhibiting 